In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import math
import scipy.stats as s
import pickle

In [ ]:
data = pd.read_csv("college_student_placement_dataset.csv")

In [ ]:
data.head()

In [ ]:
no_yes_dict = {"No":0,"Yes":1}
data[data.columns[5]] = data[data.columns[5]].map(no_yes_dict)
data[data.columns[-1]] =data[data.columns[-1]].map(no_yes_dict)

In [ ]:
data[data.columns[0]] = data[data.columns[0]].apply(lambda x: int(x[3:]))

In [ ]:
data.head()

In [ ]:
corr_matrix = data.corr()

In [ ]:
corr_matrix

In [ ]:
feature_selected_data = data[[data.columns[1],data.columns[2],
                             data.columns[3],data.columns[7],
                             data.columns[8],data.columns[9]]]

In [ ]:
feature_selected_data.head()

In [ ]:
placement_equals_1_prior = data[data[data.columns[-1]] == 1].shape[0]/data.shape[0]

In [ ]:
placement_equals_1_prior

In [ ]:
placement_equals_0_prior = (1 - placement_equals_1_prior)

In [ ]:
sns.heatmap(feature_selected_data.corr(),annot=True)

In [ ]:
sns.heatmap(feature_selected_data.iloc[:,0:-1].corr(),annot=True)

In [ ]:
X = np.array(feature_selected_data.iloc[:,0:-1])
cov_mat = np.cov(X,rowvar=False)
F = np.linalg.svd(cov_mat)

In [ ]:
E = F[0]
lamda = F[1]
new_X = np.matmul(X,F[0])
new_cov_mat = np.cov(new_X,rowvar=False)

In [ ]:
new_input_features = pd.DataFrame(data=new_X,
                            columns=["Input_feat_1", "Input_feat_2","Input_feat_3",
                                     "Input_feat_4","Input_feat_5"])

In [ ]:
new_input_features

In [ ]:
sns.heatmap(new_input_features.corr(),annot=True)

In [ ]:
new_data = pd.concat([new_input_features,feature_selected_data[data.columns[-1]]],
                     axis=1)

In [ ]:
new_data

In [ ]:
plt.hist(new_data[new_data[new_data.columns[-1]] == 1][new_data.columns[0]])

In [ ]:
new_data[new_data[new_data.columns[-1]] == 1][new_data.columns[0]]

In [ ]:
class GaussianMLEstimatorNN(torch.nn.Module):

    def __init__(self):
        super().__init__()

        self.mu = torch.nn.Parameter(data=torch.tensor([-90.0]))
        self.log_sigma = torch.nn.Parameter(data=torch.tensor([1.0]))

    def forward(self,x):

        g = -self.mu
        h = (x + g)
        i = h**2
        sigma = torch.exp(self.log_sigma)
        j = (-1/(2*sigma**2))*i
        k = torch.exp(j)
        l = (1/(math.sqrt(2*math.pi)*sigma))*k
        f = torch.log(l)

        return -torch.mean(f)

In [ ]:
def determine_accurate_params_values(input_feature_name,category):

    MLEstimatorNN = GaussianMLEstimatorNN()
    optimizer = torch.optim.SGD(params=MLEstimatorNN.parameters(),lr=0.01)
    x = torch.tensor(new_data[new_data[new_data.columns[-1]] == category][input_feature_name].values)
    tol = 10**(-4)
    model_parameters = list()

    epoch = 0
    while True:

        initial_loss_function_value = MLEstimatorNN(x)
        optimizer.zero_grad()
        initial_loss_function_value.backward()
        optimizer.step()
        final_loss_function_value = MLEstimatorNN(x)

        if torch.abs(initial_loss_function_value - final_loss_function_value) < tol:
            break

        epoch += 1

        print("The value of the Gaussian NLL after Epoch # {} is {}".format(epoch,initial_loss_function_value.item()))

    for param in MLEstimatorNN.parameters():
        model_parameters.append(param.detach().numpy().item())

    return model_parameters

In [ ]:
likelihood_distribution_params = {0:dict(),1:dict()}

for category in new_data[new_data.columns[-1]].unique():
    for input_feat in new_data.columns[0:-1]:

        likelihood_distribution_params[category][input_feat] = determine_accurate_params_values(input_feat,category)

In [ ]:
likelihood_distribution_params

In [ ]:
with open("likelihood_distribution_params.pkl", "wb") as file_handle:
    pickle.dump(likelihood_distribution_params,file_handle)

In [ ]:
np.save("eigen_vectors.npy",E)

### Now, that we know the accurate value of parameters of: $P(\text{Input\_feat\_i}|y=\text{category})$ for all $i'\text{s}$ as well as different categories($y=0$ as well as $y=1$), we can easily compute the Posterior Probabilities, $P(y=0|\vec{x})$ and $P(y=1|\vec{x})$ for the $\vec{x}$ input by the student on our webpage. 

In [ ]:
def determine_placement_posterior_probability(input_features):

    input_features = np.array(input_features)
    input_features = input_features.reshape(1,input_features.shape[0])
    eig_vectors = np.load("eigen_vectors.npy")
    new_input_features = np.matmul(input_features,eig_vectors)

    placement_equals_1_likelihood = 1.0
    placement_equals_0_likelihood = 1.0

    with open("likelihood_distribution_params.pkl","rb") as file_handle:
        likelihood_distribution_params = pickle.load(file_handle)

    for input_feat, input_feat_value in zip(new_data.columns[0:-1], new_input_features):
        mu_0, sigma_0 = likelihood_distribution_params[0][input_feat]
        mu_1, sigma_1 = likelihood_distribution_params[1][input_feat]
        
        p_input_feature_on_0_placement = s.norm.pdf(input_feat_value,mu_0,sigma_0)
        p_input_feature_on_1_placement = s.norm.pdf(input_feat_value,mu_1,sigma_1)

        placement_equals_0_likelihood = placement_equals_0_likelihood * p_input_feature_on_0_placement
        placement_equals_1_likelihood = placement_equals_1_likelihood * p_input_feature_on_1_placement

    placement_equals_0_posterior = placement_equals_0_likelihood * placement_equals_0_prior
    placement_equals_1_posterior = placement_equals_1_likelihood * placement_equals_1_prior

    if placement_equals_1_posterior[0] > placement_equals_0_posterior[0]:
        return {"result":"Given your inputs, most likeliy you are going to get placed and the probability of you getting placed is roughly {}".format(placement_equals_1_posterior)}
    else:
        return {"result":"Given your inputs, most likely you are not going to get placed and the probability of you getting placed is roughly {}".format(placement_equals_1_posterior)}
    

In [ ]:
determine_placement_posterior_probability([100,5.87,6.93,6,4])